### Compute Metrics -- LPR & WPR

Computes Line Pass Rate (LPR) and Word Pass Rate (WPR) over every `outputs/{model}.csv`
produced by `Open_Source_Models.ipynb` / `Closed_Source_Models.ipynb`, using
`compute_metrics.py` -- a port of the upstream language-confusion repo's
`compute_metrics.py`, adapted to use GlotLID instead of fastText's `lid.176` (see that
module's docstring for why, including the AfroLID literature comparison that informed
the choice).

**LPR** = fraction of completions with every line correctly identified as the target
language. **WPR** = fraction of language-correct completions with no English dictionary
word leakage -- only computed for non-Latin-script languages in our set (Amharic,
Tigrinya), matching the original repo's rationale that WPR is unreliable for
Latin-script languages (loanwords/cognates cause false positives).

### Setup

Run once, from the `african-language-confusion/` folder, in a virtual environment (keeps
these dependencies out of your global Python install):

```bash
python -m venv venv
venv\Scripts\activate      # Windows
source venv/bin/activate    # macOS/Linux
pip install -r requirements.txt
```

Then pick `venv` as this notebook's kernel before running the cells below.

First run downloads two things automatically, cached afterward: GlotLID's language-ID
model (~1.6GB, via `huggingface_hub`) and an English word list used for WPR (~1MB, saved
as `words` in this folder).

In [2]:
import glob
import itertools

import pandas as pd

import compute_metrics as cm

### Compute metrics for every output file

In [4]:
rows = []
for path in sorted(glob.glob("outputs/*.csv")):
    df = pd.read_csv(path)
    group_key = lambda o: (o["task"], o["model"])
    outputs = sorted(df.to_dict("records"), key=group_key)
    for (task, model), grouped in itertools.groupby(outputs, key=group_key):
        all_metrics = cm.compute_all_metrics(list(grouped))
        for (source, lang), metrics in all_metrics.items():
            rows.append({
                "task": task,
                "model": model,
                "source": source,
                "language": lang,
                "lpr": metrics.get("lpr"),
                "wpr": metrics.get("wpr"),
            })

results = pd.DataFrame(rows)
results

,task,model,source,language,lpr,wpr
0,crosslingual,qwen,okapi,igbo,1.000000,None
1,crosslingual,qwen,sharegpt,igbo,0.333333,None
2,crosslingual,qwen,okapi,all,1.000000,None
3,crosslingual,qwen,sharegpt,all,0.333333,None
4,crosslingual,qwen,all,igbo,0.666667,None
5,crosslingual,qwen,all,all,0.666667,None
6,monolingual,qwen,dolly,igbo,1.000000,None
7,monolingual,qwen,dolly,all,1.000000,None
8,monolingual,qwen,all,igbo,1.000000,None
9,monolingual,qwen,all,all,1.000000,None


### Save results

In [ ]:
results.to_csv("outputs/metrics_summary.csv", index=False)
print(f"Saved {len(results)} rows to outputs/metrics_summary.csv")